# Plot Constructor Example

This notebook demonstrates a decomposed pipeline: user parameters -> data loading/preprocessing -> plotting.

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
from typing import Any
import numpy as np
import pandas as pd

from app.visualization import PlotConstructor

## 1) User parameters

Set date, ionosonde station codes, and which plots should be built.

In [ ]:
# --- User-configurable parameters ---
USER_PARAMS = {
    "date_str": "2025-11-12",
    "ionosonde_codes": ["mau", "arl"],
    "plot_requests": [
        "ROTI",
        "Dst",
        "Kp",
    ],
}

USER_PARAMS

## 2) Data preparation pipeline

A small decomposed pipeline that downloads/loads only required data and returns `processor_results` for `PlotConstructor`.

In [ ]:
from app.gfz.gfz_processor import GfzProcessor
from app.kyoto.kyoto_dst_processor import KyotoProcessor
from app.omni.omni_processor import OmniProcessor
from app.simurg.simurg_processor import SimurgProcessor, DataProduct


def _fallback_results(date_str: str) -> dict[str, Any]:
    kp_df = pd.DataFrame({"datetime": pd.date_range(date_str, periods=24, freq="1h"), "kp": np.random.randint(0, 9, 24)})
    dst_df = pd.DataFrame({"datetime": pd.date_range(date_str, periods=24, freq="1h"), "dst": np.random.normal(-30, 20, 24)})
    omni_df = pd.DataFrame({"DateTime": pd.date_range(date_str, periods=24, freq="1h"), "bz": np.random.normal(0, 4, 24), "symh": np.random.normal(-20, 15, 24)})

    ts = datetime.fromisoformat(f"{date_str}T00:00:00").replace(tzinfo=timezone.utc)
    lats = np.linspace(-80, 80, 24)
    lons = np.linspace(-180, 180, 36)
    ll = np.array(np.meshgrid(lats, lons)).reshape(2, -1).T
    vals = np.random.rand(ll.shape[0])
    roti_arr = np.zeros(ll.shape[0], dtype=[("lat", "f8"), ("lon", "f8"), ("vals", "f8")])
    roti_arr["lat"] = ll[:, 0]
    roti_arr["lon"] = ll[:, 1]
    roti_arr["vals"] = vals
    roti_map = {ts: roti_arr}

    return {"ROTI": roti_map, "Kp": kp_df, "Dst": dst_df, "OMNI": omni_df}


def load_requested_results(params: dict[str, Any]) -> dict[str, Any]:
    date_str = params["date_str"]
    requested = {str(item["name"] if isinstance(item, dict) else item).strip().lower() for item in params["plot_requests"]}

    base_dir = Path.cwd().parent
    download_dir = base_dir / "files" / date_str

    results: dict[str, Any] = {}

    if "kp" in requested:
        results["Kp"] = GfzProcessor(str(download_dir / "kp")).load(date_str=date_str)

    if "dst" in requested:
        results["Dst"] = KyotoProcessor(str(download_dir / "dst")).load(date_str=date_str)

    if {"bz", "symh", "omni"} & requested:
        results["OMNI"] = OmniProcessor(str(download_dir / "omni")).load(date_str=date_str)

    if {"roti", "adjusted tec", "adjusted_tec"} & requested:
        results["ROTI"] = SimurgProcessor(str(download_dir / "simurg")).load(date_str, product_type=DataProduct.ROTI)

    # fallback for demo environments without files/network
    if any(v is None for v in results.values()) or not results:
        return _fallback_results(date_str)

    return results


processor_results = load_requested_results(USER_PARAMS)
processor_results.keys()

## 3) Build plots with PlotConstructor

In [ ]:
plotter = PlotConstructor(processor_results)
plotter.available_plots()

In [ ]:
fig, _ = plotter.plot(USER_PARAMS["plot_requests"])
fig

## 4) Optional: per-plot parameters

In [ ]:
if "ROTI" in processor_results and isinstance(processor_results["ROTI"], dict):
    first_roti_time = sorted(processor_results["ROTI"].keys())[0]
    fig, _ = plotter.plot([
        {"name": "ROTI", "params": {"plot_time": first_roti_time, "cmap": "viridis", "s": 10}},
        {"name": "Kp"},
        {"name": "Dst", "params": {"color": "black"}},
    ])
    fig